In [1]:
!pip install pyrosettacolabsetup
import pyrosettacolabsetup; pyrosettacolabsetup.install_pyrosetta(cache_wheel_on_google_drive=False, serialization=False, mirror=0)
import pyrosetta; pyrosetta.init()
from pyrosetta import *
from pyrosetta.teaching import *


Note that USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRE PURCHASE OF A LICENSE.
See https://github.com/RosettaCommons/rosetta/blob/main/LICENSE.md or email license@uw.edu for details.

Looking for compatible PyRosetta wheel file at google-drive/PyRosetta/colab.bin//wheels...

Resolving west.rosettacommons.org (west.rosettacommons.org)... 128.95.160.135, 128.95.160.134, 2607:4000:406::160:135, ...
Connecting to west.rosettacommons.org (west.rosettacommons.org)|128.95.160.135|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1665598342 (1.6G) [application/octet-stream]
--2026-09-17 14:21:23--  https://west.rosettacommons.org/pyrosetta/quarterly/release/pyrosetta-0-cp313-cp313-linux_x86_64.whl

HTTP request sent, awaiting response... 200 OK
Length: 1665598342 (1.6G) [application/octet-stream]
Saving to: ‘/PyRosetta/colab.bin/tmpz601uirr/pyrosetta-0-cp313-cp313-linux_x86_64.whl’

pyrosetta-0-cp313-c 100%[===================>]   1.55G  45.2MB/s    in 36s     

202

In [2]:
import os
import shutil

import numpy as np
import pandas as pd

import pyrosetta
from pyrosetta import rosetta

import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import files

In [3]:
pyrosetta.init("-mute all")

┌───────────────────────────────────────────────────────────────────────────────┐
│                                  PyRosetta-4                                  │
│               Created in JHU by Sergey Lyskov and PyRosetta Team              │
│               (C) Copyright Rosetta Commons Member Institutions               │
│                                                                               │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRES PURCHASE OF A LICENSE │
│          See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└───────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2026 [Rosetta PyRosetta4.Release.python313.ubuntu 2026.29+releasequarterly.80a0635615099e1b918474a63acba7b1de6fd107 2026-07-14T16:24:11] retrieved from: http://www.pyrosetta.org
core.init: Checking for fconfig files in pwd and ./rosetta/flags
core.init: Rosetta version: PyRosetta4.Release.python313.ubuntu r2 2026.29+releasequarter

In [4]:
pdb_path = None
original_pdb_name = None

upload_button = widgets.Button(
    description="Upload PDB",
    button_style="info",
    icon="upload"
)

upload_output = widgets.Output()


def upload_pdb(_):

    global pdb_path
    global original_pdb_name

    with upload_output:

        clear_output()

        uploaded = files.upload()

        pdbs = [
            name
            for name in uploaded.keys()
            if name.lower().endswith(".pdb")
        ]

        if not pdbs:
            print("Please upload a PDB file.")
            return

        original_pdb_name = pdbs[0]

        pdb_path = "/content/design_input.pdb"

        shutil.copy2(
            original_pdb_name,
            pdb_path
        )

        print(
            f"Uploaded: {original_pdb_name}"
        )

        print(
            f"Internal path: {pdb_path}"
        )


upload_button.on_click(upload_pdb)

display(
    upload_button,
    upload_output
)

Button(button_style='info', description='Upload PDB', icon='upload', style=ButtonStyle())

Output()

In [6]:
pose = None

def load_pose():

    global pose

    if pdb_path is None:
        raise ValueError(
            "Please upload a PDB first."
        )

    pose = pyrosetta.pose_from_pdb(
        pdb_path
    )

    print(
        f"Loaded {pose.total_residue()} residues."
    )

    print(
        f"Sequence length: {len(pose.sequence())}"
    )

    return pose

In [7]:
pose = load_pose()

Loaded 182 residues.
Sequence length: 182


In [8]:
ligand_widget = widgets.Text(
    value="LIG",
    description="Ligand:",
    layout=widgets.Layout(width="300px")
)

display(ligand_widget)

Text(value='LIG', description='Ligand:', layout=Layout(width='300px'))

In [10]:
print("Non-protein residues:")
print("-" * 60)

for i in range(
    1,
    pose.total_residue() + 1
):

    residue = pose.residue(i)

    if not residue.is_protein():

        print(
            f"Pose {i:4d} | "
            f"{residue.name3():<6} | "
            f"PDB {pose.pdb_info().pose2pdb(i)}"
        )

Non-protein residues:
------------------------------------------------------------
Pose  181 |  MG    | PDB 616 B 
Pose  182 | ADP    | PDB 616 C 


In [11]:
fixed_residues_widget = widgets.Textarea(
    value="",
    placeholder=(
        "One residue per line\n"
        "Example:\n"
        "A:45\n"
        "A:78\n"
        "A:102"
    ),
    description="Fixed residues:",
    layout=widgets.Layout(
        width="450px",
        height="130px"
    )
)

display(fixed_residues_widget)

Textarea(value='', description='Fixed residues:', layout=Layout(height='130px', width='450px'), placeholder='O…

In [12]:
contact_distance_widget = widgets.FloatSlider(
    value=4.0,
    min=2.5,
    max=8.0,
    step=0.5,
    description="Ligand contact (Å):",
    continuous_update=False
)

display(contact_distance_widget)

FloatSlider(value=4.0, continuous_update=False, description='Ligand contact (Å):', max=8.0, min=2.5, step=0.5)

In [13]:
def parse_fixed_residues(text):

    fixed = []

    for line in text.strip().splitlines():

        line = line.strip()

        if not line or line.startswith("#"):
            continue

        parts = line.split(":")

        if len(parts) != 2:
            raise ValueError(
                f"Invalid residue specification: {line}\n"
                "Use Chain:PDB_residue, e.g. A:45"
            )

        chain = parts[0].strip()
        pdb_resnum = int(
            parts[1].strip()
        )

        fixed.append({
            "chain": chain,
            "pdb_resnum": pdb_resnum
        })

    return fixed

In [20]:
def convert_fixed_residues(
    pose,
    fixed_residues
):

    pose_numbers = []

    for residue in fixed_residues:

        pose_num = pose.pdb_info().pdb2pose(
            residue["chain"],
            residue["pdb_resnum"]
        )

        if pose_num == 0:

            raise ValueError(
                f"Could not find "
                f"{residue['chain']}:"
                f"{residue['pdb_resnum']} "
                f"in the structure."
            )

        pose_numbers.append(
            pose_num
        )

    return pose_numbers

In [21]:
fixed_residues = parse_fixed_residues(
    fixed_residues_widget.value
)

fixed_pose_numbers = convert_fixed_residues(
    pose,
    fixed_residues
)

print(
    "Fixed Pose residues:",
    fixed_pose_numbers
)

Fixed Pose residues: []


In [15]:
def find_ligand_residues(
    pose,
    ligand_name
):

    ligand_name = ligand_name.strip().upper()

    ligand_residues = []

    for i in range(
        1,
        pose.total_residue() + 1
    ):

        residue = pose.residue(i)

        if (
            residue.name3()
            .strip()
            .upper()
            == ligand_name
        ):

            ligand_residues.append(i)

    if not ligand_residues:

        raise ValueError(
            f"Ligand '{ligand_name}' "
            "was not found in the PDB."
        )

    return ligand_residues

In [16]:
def find_ligand_contacts(
    pose,
    ligand_residues,
    cutoff=4.0
):

    contact_residues = set()

    for ligand_resnum in ligand_residues:

        ligand = pose.residue(
            ligand_resnum
        )

        for protein_resnum in range(
            1,
            pose.total_residue() + 1
        ):

            protein_residue = pose.residue(
                protein_resnum
            )

            if not protein_residue.is_protein():
                continue

            min_distance = float("inf")

            for lig_atom in range(
                1,
                ligand.nheavyatoms() + 1
            ):

                lig_xyz = ligand.xyz(
                    lig_atom
                )

                for prot_atom in range(
                    1,
                    protein_residue.nheavyatoms()
                    + 1
                ):

                    prot_xyz = (
                        protein_residue.xyz(
                            prot_atom
                        )
                    )

                    distance = (
                        lig_xyz - prot_xyz
                    ).norm()

                    if distance < min_distance:
                        min_distance = distance

            if min_distance <= cutoff:

                contact_residues.add(
                    protein_resnum
                )

    return sorted(
        contact_residues
    )

In [17]:
ligand_residues = find_ligand_residues(
    pose,
    ligand_widget.value
)

contact_residues = find_ligand_contacts(
    pose,
    ligand_residues,
    cutoff=contact_distance_widget.value
)

print(
    f"Ligand residues: {ligand_residues}"
)

print(
    f"\nProtein residues within "
    f"{contact_distance_widget.value:.1f} Å:"
)

for resnum in contact_residues:

    print(
        f"Pose {resnum:4d} | "
        f"{pose.residue(resnum).name3():<3} | "
        f"PDB {pose.pdb_info().pose2pdb(resnum)}"
    )

Ligand residues: [182]

Protein residues within 5.0 Å:
Pose   85 | GLU | PDB 85 A 
Pose   89 | ALA | PDB 89 A 
Pose  150 | GLN | PDB 150 A 
Pose  153 | LEU | PDB 153 A 
Pose  154 | SER | PDB 154 A 
Pose  157 | GLY | PDB 157 A 
Pose  170 | ASN | PDB 170 A 
Pose  173 | ARG | PDB 173 A 
Pose  174 | LEU | PDB 174 A 
Pose  177 | ARG | PDB 177 A 


In [24]:
def run_fastdesign(
    pose,
    fixed_pose_numbers,
    contact_pose_numbers
):

    # --------------------------------------------------
    # Score function
    # --------------------------------------------------

    scorefxn = pyrosetta.create_score_function(
        "ref2015"
    )

    # --------------------------------------------------
    # TaskFactory
    # --------------------------------------------------

    task_factory = (
        rosetta.core.pack.task.TaskFactory()
    )

    task_factory.push_back(
        rosetta.core.pack.task.operation
        .InitializeFromCommandline()
    )

    # --------------------------------------------------
    # Completely freeze fixed/catalytic residues
    # --------------------------------------------------

    if fixed_pose_numbers:

        # ResidueIndexSelector wants a string such as
        # "12,34,57"
        fixed_string = ",".join(
            str(x)
            for x in fixed_pose_numbers
        )

        fixed_selector = (
            rosetta.core.select.residue_selector
            .ResidueIndexSelector(
                fixed_string
            )
        )

        prevent_repacking = (
            rosetta.core.pack.task.operation
            .PreventRepackingRLT()
        )

        task_factory.push_back(
            rosetta.core.pack.task.operation
            .OperateOnResidueSubset(
                prevent_repacking,
                fixed_selector
            )
        )

    # --------------------------------------------------
    # FastDesign
    # --------------------------------------------------

    fastdesign = (
        rosetta.protocols.denovo_design
        .movers.FastDesign()
    )

    fastdesign.set_scorefxn(
        scorefxn
    )

    fastdesign.set_task_factory(
        task_factory
    )

    # --------------------------------------------------
    # MoveMap
    # --------------------------------------------------

    movemap = pyrosetta.MoveMap()

    # Allow movement everywhere initially
    movemap.set_bb(True)
    movemap.set_chi(True)

    # Don't move chains relative to one another
    movemap.set_jump(False)

    # Completely freeze fixed/catalytic residues
    for resnum in fixed_pose_numbers:

        movemap.set_bb(
            resnum,
            False
        )

        movemap.set_chi(
            resnum,
            False
        )

    fastdesign.set_movemap(
        movemap
    )

    # --------------------------------------------------
    # Run
    # --------------------------------------------------

    start_score = float(
        scorefxn(pose)
    )

    fastdesign.apply(pose)

    final_score = float(
        scorefxn(pose)
    )

    return (
        pose,
        start_score,
        final_score
    )

In [25]:
run_button = widgets.Button(
    description="Run FastDesign",
    button_style="success",
    icon="play"
)

run_output = widgets.Output()

designed_pose = None


def run_design(_):

    global designed_pose

    with run_output:

        clear_output()

        try:

            if pose is None:
                raise ValueError(
                    "No PDB has been loaded."
                )

            # ------------------------------------------
            # Fixed residues
            # ------------------------------------------

            fixed_residues = (
                parse_fixed_residues(
                    fixed_residues_widget.value
                )
            )

            fixed_pose_numbers = (
                convert_fixed_residues(
                    pose,
                    fixed_residues
                )
            )

            # ------------------------------------------
            # Ligand
            # ------------------------------------------

            ligand_residues = (
                find_ligand_residues(
                    pose,
                    ligand_widget.value
                )
            )

            # ------------------------------------------
            # Contacts
            # ------------------------------------------

            contact_residues = (
                find_ligand_contacts(
                    pose,
                    ligand_residues,
                    cutoff=(
                        contact_distance_widget.value
                    )
                )
            )

            # ------------------------------------------
            # Report setup
            # ------------------------------------------

            print("=" * 70)
            print("FASTDESIGN")
            print("=" * 70)

            print(
                f"Ligand: "
                f"{ligand_widget.value}"
            )

            print(
                f"Ligand residue(s): "
                f"{ligand_residues}"
            )

            print()

            print(
                "Fixed/catalytic residues:"
            )

            for resnum in fixed_pose_numbers:

                print(
                    f"  {resnum}: "
                    f"{pose.residue(resnum).name3()} "
                    f"({pose.pdb_info().pose2pdb(resnum)})"
                )

            print()

            print(
                f"Ligand-contact residues "
                f"({contact_distance_widget.value:.1f} Å):"
            )

            for resnum in contact_residues:

                print(
                    f"  {resnum}: "
                    f"{pose.residue(resnum).name3()} "
                    f"({pose.pdb_info().pose2pdb(resnum)})"
                )

            print()

            # ------------------------------------------
            # Run FastDesign
            # ------------------------------------------

            design_pose = pose.clone()

            print(
                "Running FastDesign..."
            )

            (
                designed_pose,
                start_score,
                final_score
            ) = run_fastdesign(
                design_pose,
                fixed_pose_numbers,
                contact_residues
            )

            print(
                "FastDesign complete."
            )

            print()

            print(
                f"Starting score: "
                f"{start_score:.3f}"
            )

            print(
                f"Final score: "
                f"{final_score:.3f}"
            )

            print(
                f"Score change: "
                f"{final_score - start_score:.3f}"
            )

        except Exception as e:

            print(
                "ERROR:",
                type(e).__name__,
                e
            )


run_button.on_click(
    run_design
)

display(
    run_button,
    run_output
)

Button(button_style='success', description='Run FastDesign', icon='play', style=ButtonStyle())

Output()

In [ ]:
def verify_fixed_residues(
    original_pose,
    designed_pose,
    fixed_pose_numbers
):

    rows = []

    for resnum in fixed_pose_numbers:

        original_aa = (
            original_pose
            .residue(resnum)
            .name3()
        )

        designed_aa = (
            designed_pose
            .residue(resnum)
            .name3()
        )

        rows.append({
            "Pose_residue": resnum,
            "PDB_residue":
                original_pose
                .pdb_info()
                .pose2pdb(resnum),
            "Original": original_aa,
            "Designed": designed_aa,
            "Identity_preserved":
                original_aa == designed_aa
        })

    return pd.DataFrame(rows)

In [ ]:
fixed_check = verify_fixed_residues(
    pose,
    designed_pose,
    fixed_pose_numbers
)

display(
    fixed_check
)

In [ ]:
save_button = widgets.Button(
    description="Save designed PDB",
    button_style="warning",
    icon="save"
)

save_output = widgets.Output()


def save_design(_):

    with save_output:

        clear_output()

        if designed_pose is None:

            print(
                "Run FastDesign first."
            )

            return

        output_path = (
            "/content/"
            "fastdesign_output.pdb"
        )

        designed_pose.dump_pdb(
            output_path
        )

        print(
            f"Saved: {output_path}"
        )


save_button.on_click(
    save_design
)

display(
    save_button,
    save_output
)

In [ ]:
download_button = widgets.Button(
    description="Download designed PDB",
    button_style="info",
    icon="download"
)

download_output = widgets.Output()


def download_design(_):

    with download_output:

        clear_output()

        output_path = (
            "/content/"
            "fastdesign_output.pdb"
        )

        if not os.path.exists(
            output_path
        ):

            print(
                "Save the designed PDB first."
            )

            return

        files.download(
            output_path
        )


download_button.on_click(
    download_design
)

display(
    download_button,
    download_output
)